# Demand Analysis & Forecasting Pipeline

End-to-end walkthrough of the AI demand sensing engine.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.data.synthetic_generator import generate_synthetic_data

data = generate_synthetic_data(n_skus=20, n_weeks=104)
sales = data['sales']
print(f'Sales data: {sales.shape}')
sales.head()

In [ ]:
# Demand distribution by SKU
sku_summary = sales.groupby('group_id')['demand'].agg(['mean', 'std', 'sum']).round(1)
sku_summary['cv'] = (sku_summary['std'] / sku_summary['mean']).round(3)
sku_summary.sort_values('sum', ascending=False).head(10)

In [ ]:
# ABC/XYZ Segmentation
from src.optimization.segmentation import ABCXYZSegmenter

seg = ABCXYZSegmenter({'abc_thresholds': [0.80, 0.95], 'xyz_thresholds': [0.5, 1.0]})
result = seg.classify(sales)
result[['group_id', 'abc_class', 'xyz_class', 'segment', 'policy']].head(10)

In [ ]:
# Safety Stock Calculation
from src.optimization.safety_stock import SafetyStockCalculator

calc = SafetyStockCalculator({'target_service_level': 0.98, 'method': 'demand_variability'})
ss = calc.calculate(sales, data['lead_times'])
ss[['group_id', 'safety_stock_units', 'avg_daily_demand', 'z_score']].head(10)

In [ ]:
# Reorder Points
from src.optimization.reorder_point import ReorderPointCalculator

rop = ReorderPointCalculator({})
rop_result = rop.calculate(ss, data['unit_costs'])
rop_result[['group_id', 'reorder_point', 'eoq', 'inventory_investment']].head(10)

In [ ]:
print(f"Total inventory investment: ${rop_result['inventory_investment'].sum():,.0f}")
print(f"Average service level: 98%")
print(f"SKU segments: {result['segment'].value_counts().to_dict()}")